In [ ]:
import torch
import torch.nn as nn
from models.multimodal import MultiModalRegressor
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data.datamodule import DataModule

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiModalRegressor()
checkpoint = torch.load("checkpoints/ATT&DAR_MULTIMODAL_2025-05-18_10-50-05.pt",weights_only=False)["model_state_dict"]
model.load_state_dict(checkpoint, strict=False)
model.to(device)

Using cache found in /users/eleves-b/2023/keyvan.attarian/.cache/torch/hub/facebookresearch_dinov2_main


shape of image encoder:  768
shape of text encoder:  768
shape of fusion:  1536


MultiModalRegressor(
  (image_encoder): DinoV2Finetune(
    (backbone): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (blocks): ModuleList(
        (0-11): 12 x NestedTensorBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): MemEffAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (act): GELU(approximate='none')
            (fc2): Linear(in_features=3072, ou

In [14]:
content = pd.read_csv ("dataset/train_val.csv")
test_content = pd.read_csv ("dataset/test.csv")

In [ ]:
datamodule = Data
full_loader = datamodule.test_dataloader()

resultats = []

with torch.no_grad():
    # Attention : les DataLoader ne sont pas indexables directement
    for batch in full_loader:
        batch["image"] = batch["image"].to(device)
        batch["channel"] = batch["channel"].to(device)
        batch["year"] = batch["year"].to(device)
        with torch.no_grad():
            out_data = model(batch)
            out_data = torch.expm1(out_data)
        resultats.append({"ID" : batch["id"].detach().cpu().numpy()[0], "TARGET" : out_data.detach().cpu().numpy()[0][0]})